**Linda Zier**

**ST 554 ~ Final Project**

**04/30/2026**

**Goal**

For this project we :

*   added our project and files to our github repo, committing often to show our progress.
*   wrote a Jupyter notebook that fits a machine learning model using pyspark’s MLlib module. In that same notebook we wrote code to read in a stream of data (data that we produced ourselves using a .py file that is also kept in the repo).
*   we used the model to do predictions on the stream and wrote those out to the console.


**Data**

The data is modified from the UCI machine learning repository. The file power_ml_data.csv is available at the URL: https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv. The study was about relating power consumption from different zones of Tetouan city to various factors like time of day, temperature, and
humidity.


*   We used a chunk to build our model.
*   We then 'streamed data' to a folder that we monitored. As data came in we used our fitted model to predict on the incoming data.





# Fitting the Model

We created a Jupyter notebook for the model fitting part and the streaming part below. We completed the following:

*   read the data into a standard pandas data frame using the pd.read_csv() function
*   converted this to a spark data frame
*   treated the Power_Zone_3 variable as our response variable and used the other variables as predictors

In [1]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, VectorAssembler, Binarizer, \
                               OneHotEncoder, PCA
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql.functions import col
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder.getOrCreate()

# read in data/power_ml_data.csv as pandas dataframe
powerDF=pd.read_csv("data/power_ml_data.csv")

#convert to spark dataframe
powerDF=spark.createDataFrame(powerDF)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/29 16:19:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Creating the Pipeline

We fit an elastic net model using CV with the steps below.
The transformations below each used an MLlib function that we put into a pipeline.

*   We used an SQL transformer to cast the hour variable as a DoubleType.

*   We binarized the Hour column based on the column being less than 6.5 or not (night vs day essentially).

*   The month column was one-hot encoded.
*   We Ran a PCA fit on the Temperature, Humidity, Wind_Speed, General_Diffuse_Flows, and Diffuse_Flows columns. We did this by:
    - first by using a VectorAssembler() call to place these variables in a column together for use with the PCA() estimator
    
    - then we had a PCA transformer for use in our pipeline.
    - we used two PCs in our transformation.


*   We renamed our response variable as label

*   We used VectorAssembler() to put our predictors into a features. The predictors are:

    – two fitted PCA features

    – binary Hour variable

    – Power_Zone_1

    – Power_Zone_2

    – Month indicator variables


In [2]:
# check the data types
powerDF.printSchema()

# cast the hour as double since it is a long
sql = SQLTransformer(statement = '''
                     SELECT *, 
                     CAST(Hour AS DOUBLE) AS HourD FROM __THIS__
                     ''')

# binarize night vs day
binarizer = Binarizer(threshold=6.5, inputCol="HourD", outputCol="Hour_bin")

# one hot encode month
ohe = OneHotEncoder(inputCols=["Month"], outputCols=["Month_ohe"])

#VectorAssembler to bundle features together for pca
pca_assembler = VectorAssembler(
    inputCols=["Temperature", "Humidity", "Wind_Speed", 
               "General_Diffuse_Flows", "Diffuse_Flows"],
    outputCol="pca_input")

# pca with 2 components
pca = PCA(k=2, inputCol="pca_input", outputCol="pca_features")

# response variable to label
sql_label= SQLTransformer(statement = '''
                          SELECT *,
                          Power_Zone_3 AS label FROM __THIS__
                          ''')
# assemble final features
assembler = VectorAssembler(
    inputCols=["pca_features", "Hour_bin", "Power_Zone_1", 
               "Power_Zone_2","Month_ohe"],
    outputCol="features")

print("TRANSFORMATIONS COMPLETE")   
                     

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)

TRANSFORMATIONS COMPLETE


In [3]:
from pyspark.ml import Pipeline

#build pipeline
pipeline= Pipeline(stages = [sql, binarizer, ohe, pca_assembler, 
                             pca, sql_label, assembler])
fittedPipeline = pipeline.fit(powerDF)
transformedDF=fittedPipeline.transform(powerDF)

print("PIPELINE COMPLETE")

PIPELINE COMPLETE


### Fitting an Elastic Net Model
Next we used the CrossValidator() function and the LinearRegression() function to fit an elastic net model. We did multiple combinations of reg and elastic net parameters.  We fit the model using 5-fold cross validation with root mean square error (RMSE) as the criteria: we're training 5 separate models (one per fold) and averaging their RMSE's together to get their RMSE for that combination. We then report the optimal values chosen for the tuning parameters and the CV error which is the RMSE from the best model.


In [4]:

# setting up elastic net model
lr= LinearRegression(elasticNetParam=0.5)


#  grid for the regParam and elasticNetParam
paramGrid= ParamGridBuilder() \
    .addGrid(lr.regParam,[0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr.elasticNetParam,[ 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()

# 5-fold CV with rmse evaluator
cv=CrossValidator(estimator=lr,
                   estimatorParamMaps = paramGrid,  
                   evaluator = RegressionEvaluator(metricName= 'rmse'),
                   numFolds=5)

# fit the model
cvModel=cv.fit(transformedDF)

# Report the optimal values chosen for the tuning parameters
print("Optimal regParam:", cvModel.bestModel.getRegParam())
print("Optimal elasticNetParam:", cvModel.bestModel.getElasticNetParam())

# report RMSE errors - if you want to see all 11 x 11 = 121 of them
# print("RMSE errors= ", cvModel.avgMetrics)

# report lowest RMSE
print("CV RMSE = ", min(cvModel.avgMetrics))


26/04/29 16:19:15 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/29 16:19:15 WARN Instrumentation: [4395f6d3] regParam is zero, which might cause numerical instability and overfitting.
26/04/29 16:19:17 WARN Instrumentation: [4395f6d3] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/29 16:19:18 WARN Instrumentation: [d2afac20] regParam is zero, which might cause numerical instability and overfitting.
26/04/29 16:19:18 WARN Instrumentation: [d2afac20] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/29 16:19:19 WARN Instrumentation: [80596e4f] regParam is zero, which might cause numerical instability and overfitting.
26/04/29 16:19:19 WARN Instrumentation: [80596e4f] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
2

Optimal regParam: 0.05
Optimal elasticNetParam: 0.95
CV RMSE =  2147.7287659222


We reported the training set RMSE using our fitted model as a transformer and
evaluating on the entire training set.

We then created a residual column (label - prediction) and printed the data frame with these
residuals.  I also printed a summary table as a sanity check.

In [5]:
# report training set RMSE by using fitted model as a transformer
predictions = cvModel.transform(transformedDF)
trainRMSE = RegressionEvaluator(metricName= 'rmse').evaluate(predictions)
print("Training RMSE=", trainRMSE)

# create residual column and display results
print("Training set residuals and summary:")
predictions = predictions.withColumn("residual", col("label") - col("prediction"))
predictions.select("label", "prediction", "residual").show()

#sanity check on residual distribution
predictions.select("label", "prediction", "residual") \
    .summary("mean", "stddev", "min", "max") \
    .show()

Training RMSE= 2147.097420811707
Training set residuals and summary:
+-----------+------------------+------------------+
|      label|        prediction|          residual|
+-----------+------------------+------------------+
|20240.96386| 20880.66963711528|-639.7057771152795|
|20131.08434|18659.639974645514|1471.4443653544877|
|19668.43373|18204.143168808827|1464.2905611911738|
|18899.27711|17590.078208113446|1309.1989018865534|
|18442.40964| 16996.70738645299|1445.7022535470132|
|18130.12048|16517.100323932034| 1613.020156067967|
|17945.06024| 16092.67588969123|1852.3843503087683|
|17459.27711|15722.122138411652|1737.1549715883466|
|17025.54217|15270.480543428048|1755.0616265719527|
|16794.21687|14937.778286297898|1856.4385837021018|
|16638.07229|14651.923756069167|1986.1485339308329|
|16395.18072|14414.443312220921|1980.7374077790791|
|16117.59036| 14082.31344106377|2035.2769189362298|
| 15822.6506|13624.325164293781|2198.3254357062197|
|15672.28916|13449.842621214577|2222.4465387854

# Handling Streaming Data
We downloaded a file from: https://www4.stat.ncsu.edu/~online/datasets/power_streaming_data.csv
and stored it in our final_project/data directory.  This was our source of random sampling.
### Reading a Stream
We read in a stream in the form of .csv files. I created a folder (using mkdir in terminal) called streaming_data where I will read in my .csv files. The schema is set to that of the original data since that is what our incoming data will look like and a header is assumed present.

In [10]:
# set the schema to that of the original data
stream_schema=powerDF.schema
print(stream_schema)

# set up readstream with a header
streamDF = spark.readStream.schema(stream_schema).option("header", True)\
           .csv("streaming_data")

#showing current working directory
#import os
#os.getcwd()

StructType([StructField('Temperature', DoubleType(), True), StructField('Humidity', DoubleType(), True), StructField('Wind_Speed', DoubleType(), True), StructField('General_Diffuse_Flows', DoubleType(), True), StructField('Diffuse_Flows', DoubleType(), True), StructField('Power_Zone_1', DoubleType(), True), StructField('Power_Zone_2', DoubleType(), True), StructField('Power_Zone_3', DoubleType(), True), StructField('Month', LongType(), True), StructField('Hour', LongType(), True)])



### Transform/Aggregation Step

In this code block we use our model transformer to obtain predictions from the incoming data stream. 

First we created a residual column as we did in the previous section and return only label, prediction, and residual.

Then with another transformation on the (original) stream, we modified the response variable to be called label.

Lastly we joined our two transformations based on the label variable.

In [24]:
# ---Transformation 1:---

# apply transformer to the data stream
streamPredictions = cvModel.transform(fittedPipeline.transform(streamDF))

#add residual column = label - predictions
streamResiduals=streamPredictions.withColumn("residual",col("label")- col("prediction")) \
                                  .select("label","prediction", "residual")
                                              
# ---Transformation 2:---

# rename response to
streamLabeled=sql_label.transform(streamDF)

# --- Inner Join ---
streamJoined = streamResiduals.join(streamLabeled, on="label", how="inner")



### Writing Step

 We used the append output  mode to write our stream to the console and started our query.

In [25]:
# Write stream to console in append mode and start query
#query = streamJoined.writeStream.outputMode("append").format("console")

query = streamJoined.writeStream \
                    .outputMode("append") \
                    .format("console") \
                    .start()


26/04/29 18:35:03 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-e25768b6-7f08-493a-9443-f99e2fad8bdb. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/29 18:35:03 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|9301.899898|15562.896660848064|-6260.9967628480645|      18.92|    82.6|      4.92|                193.9|        120.0| 34961.41593| 23845.32225| 9301.899898|    9|   9|
|16239.03614| 15845.99582871796|  393.0403112820404|      11.35|    57.7|     0.087|                0.048|        0.096| 25725.56962| 16249.24012| 16239.03614|    1|   0|
| 11070.6383| 9419.548296945586| 1651.0900030544

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|18426.18182|19106.075203109467| -679.8933831094655|      21.35|   43.18|     0.081|                287.9|        269.0| 33555.56512| 21343.38086| 18426.18182|    4|  17|
|15959.27273|15665.605110733084|  293.6676192669165|      13.48|    85.1|     4.916|                0.048|        0.208| 24143.63832| 12981.26273| 15959.27273|    4|   4|
|11922.26721| 12784.35496827819| -862.0877582781

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|25980.78392|24437.766427922616| 1543.0174920773861|      10.34|   53.67|     0.081|                0.176|        0.141| 41796.61017| 23883.28267| 25980.78392|    2|  19|
|30324.35146|28472.714268539297| 1851.6371914607043|      27.52|   69.04|     4.915|                401.7|        293.3| 38221.39535| 26065.82278| 30324.35146|    7|  15|
|11549.17933|12967.259178073107|-1418.0798480731

-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|9219.687875|11584.746050673197| -2365.058175673197|       16.6|   68.76|     0.073|                177.6|        138.6| 31294.29658|  24898.4351| 9219.687875|   12|  16|
|18397.09091|17978.057748893305|   419.033161106694|      22.36|   39.27|     0.074|                758.0|        127.5|  32898.3423| 20335.23422| 18397.09091|    4|  15|
|19152.73846|21116.281262882643|-1963.5428028826

-------------------------------------------
Batch: 4
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      label|        prediction|          residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|    24256.0| 22585.34444748835|1670.6555525116491|      16.29|    80.2|      0.07|                59.64|        55.72| 38379.33262| 16174.33809|     24256.0|    4|  18|
| 10594.6988| 9258.098265055343|1336.6005349446568|      12.31|    86.3|     4.917|                0.029|        0.145| 21624.61538| 15422.72727|  10594.6988|   11|   3|
|26877.99163| 25505.74018485032| 1372.251445149679|  

In [26]:
# stop querying
query.stop()

for s in spark.streams.active:
    s.stop()
    
#for f in os.listdir("streaming_data"):
#    os.remove(f"streaming_data/{f}")

26/04/29 18:39:02 WARN DAGScheduler: Failed to cancel job group e9d0c86c-7d42-4e9c-89ff-d46cfbc8a8c2. Cannot find active jobs for it.
26/04/29 18:39:02 WARN DAGScheduler: Failed to cancel job group e9d0c86c-7d42-4e9c-89ff-d46cfbc8a8c2. Cannot find active jobs for it.
